# Transitive entity clustering and name review

Run the cells in order using a Python kernel with `polars`, `scikit-learn`, and
`sparse_dot_topn` installed. This notebook exercises the standalone exact engine,
separate name diagnostics, and explicit reviewed splits.

The synthetic example is small enough to inspect end to end. The real-data section
uses the local `test_data/100k.parquet` file (968,512 rows), initially limited to
10,000 rows. Original files are read only; exports go to a separate output directory.

If dependencies are missing, install them in the selected kernel with
`%pip install polars scikit-learn sparse_dot_topn`.

In [1]:
from pathlib import Path
from time import perf_counter
from importlib.metadata import version
import json
import sys

import polars as pl
from IPython.display import display, FileLink

# Works when the notebook opens from the repository root or a subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "transitive_fusion.py").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the entity-fusion repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from transitive_fusion import cluster_entities
from transitive_review import evaluate_clusters, apply_cluster_splits

OUTPUT_DIR = ROOT / "test_data" / "_debug_outputs" / "transitive_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", sys.executable)
print("Repository:", ROOT)
print("Output directory:", OUTPUT_DIR)
print({name: version(name) for name in ("polars", "scikit-learn", "sparse_dot_topn")})


def show_file(path):
    # Relative links work with the Jupyter server's file routes.
    try:
        relative = Path(path).relative_to(Path.cwd())
    except ValueError:
        print(Path(path).resolve())
    else:
        display(FileLink(str(relative)))


Python: /Users/shumanti/Development/entity-fusion/.venv/bin/python
Repository: /Users/shumanti/Development/entity-fusion
Output directory: /Users/shumanti/Development/entity-fusion/test_data/_debug_outputs/transitive_notebook
{'polars': '1.33.1', 'scikit-learn': '1.6.1', 'sparse_dot_topn': '1.1.5'}


## 1. Synthetic records

A and B share SSN. B and C share license/phone. C and D share phone/email.
E has a missing name but shares A/B's SSN. F is a singleton. Names are deliberately
split into two families, so the exact cluster is worth inspecting.

In [2]:
toy = pl.DataFrame({
    "record_id": list("ABCDEF"),
    "name": ["NORTH HARBOR CREDIT UNION", "north harbor credit union",
             "DESERT VALLEY FINANCE", "DESERT VALLEY FINANCE", None,
             "UNRELATED INSTITUTION"],
    "ssn": ["SYNTH-1", "SYNTH-1", "SYNTH-2", "SYNTH-3", "SYNTH-1", None],
    "driver_license": ["L1", "L2", "L2", "L4", "L1", None],
    "license_state": ["NY", "NY", "NY", "CA", "NY", None],
    "phone": ["555-0101", "555-0102", "555-0102", "555-0102", "555-0101", None],
    "country": ["US", "US", "US", "US", "US", None],
    "email": ["a@example.test", "b@example.test", "c@example.test",
              "c@example.test", "a@example.test", None],
})
display(toy)


record_id,name,ssn,driver_license,license_state,phone,country,email
str,str,str,str,str,str,str,str
"""A""","""NORTH HARBOR CREDIT UNION""","""SYNTH-1""","""L1""","""NY""","""555-0101""","""US""","""a@example.test"""
"""B""","""north harbor credit union""","""SYNTH-1""","""L2""","""NY""","""555-0102""","""US""","""b@example.test"""
"""C""","""DESERT VALLEY FINANCE""","""SYNTH-2""","""L2""","""NY""","""555-0102""","""US""","""c@example.test"""
"""D""","""DESERT VALLEY FINANCE""","""SYNTH-3""","""L4""","""CA""","""555-0102""","""US""","""c@example.test"""
"""E""",null,"""SYNTH-1""","""L1""","""NY""","""555-0101""","""US""","""a@example.test"""
"""F""","""UNRELATED INSTITUTION""",null,null,null,null,null,null


## 2. Simple K-of-N matching

Compare one-of-two with two-of-two exact matches. Values match within each field;
nulls, NaNs, and blank strings do not create links. Transitivity applies after the
direct matching rule: every pair in a cluster does not have to match directly.

In [3]:
any_id = cluster_entities(toy, ["ssn", "driver_license"], min_matches=1)
both_ids = cluster_entities(toy, ["ssn", "driver_license"], min_matches=2)
display(pl.DataFrame({
    "record_id": toy["record_id"],
    "one_of_two_cluster": any_id["cluster_id"],
    "two_of_two_cluster": both_ids["cluster_id"],
}))


record_id,one_of_two_cluster,two_of_two_cluster
str,u64,u64
"""A""",0,0
"""B""",0,1
"""C""",0,2
"""D""",3,3
"""E""",0,0
"""F""",5,5


## 3. Nested rules and blockers

The rule is **SSN OR at least two of license, phone, and email**. A license
condition requires the same license state; phone requires the same country.
Blockers constrain a condition and do not count as extra votes.

Edit the rule, then rerun from this cell to explore alternatives. Use `and` to
require all children, `or` for any child, or `at_least` with `conditions`.

In [4]:
rule = {
    "or": [
        {"field": "ssn"},
        {
            "at_least": 2,
            "conditions": [
                {"field": "driver_license", "block_on": ["license_state"]},
                {"field": "phone", "block_on": ["country"]},
                {"field": "email"},
            ],
        },
    ],
}
fusion = cluster_entities(toy, match_rule=rule, return_details=True)
display(fusion.records)
display(fusion.match_groups)
assert fusion.records.drop("cluster_id").equals(toy)


record_id,name,ssn,driver_license,license_state,phone,country,email,cluster_id
str,str,str,str,str,str,str,str,u64
"""A""","""NORTH HARBOR CREDIT UNION""","""SYNTH-1""","""L1""","""NY""","""555-0101""","""US""","""a@example.test""",0
"""B""","""north harbor credit union""","""SYNTH-1""","""L2""","""NY""","""555-0102""","""US""","""b@example.test""",0
"""C""","""DESERT VALLEY FINANCE""","""SYNTH-2""","""L2""","""NY""","""555-0102""","""US""","""c@example.test""",0
"""D""","""DESERT VALLEY FINANCE""","""SYNTH-3""","""L4""","""CA""","""555-0102""","""US""","""c@example.test""",0
"""E""",null,"""SYNTH-1""","""L1""","""NY""","""555-0101""","""US""","""a@example.test""",0
"""F""","""UNRELATED INSTITUTION""",null,null,null,null,null,null,5


group_id,cluster_id,row_positions,conditions,shared_values
u64,u64,list[u64],list[str],str
0,0,"[0, 1, 4]","[""ssn""]","""{""ssn"": ""SYNTH-1""}"""
1,0,"[0, 4]","[""driver_license [block_on: license_state]"", ""email""]","""{""driver_license"": ""L1"", ""emai…"
2,0,"[0, 4]","[""driver_license [block_on: license_state]"", ""phone [block_on: country]""]","""{""country"": ""US"", ""driver_lice…"
3,0,"[0, 4]","[""email"", ""phone [block_on: country]""]","""{""country"": ""US"", ""email"": ""a@…"
4,0,"[1, 2]","[""driver_license [block_on: license_state]"", ""phone [block_on: country]""]","""{""country"": ""US"", ""driver_lice…"
5,0,"[2, 3]","[""email"", ""phone [block_on: country]""]","""{""country"": ""US"", ""email"": ""c@…"


## 4. Explain exact connections

Each match-group row represents mutually linked members and the sufficient rule
conditions/shared values. Overlapping groups explain the transitive chain.
Record positions are zero-based input row positions; the HTML uses record IDs as
additional labels. Open the report to inspect paths and true bridge links.

In [5]:
exact_report = fusion.export_html(
    OUTPUT_DIR / "synthetic_exact_review.html",
    record_id_column="record_id",
    conflict_columns=["ssn"],
    common_value_min_count=3,
)
show_file(exact_report)


/Users/shumanti/Development/entity-fusion/test_data/_debug_outputs/transitive_notebook/synthetic_exact_review.html

## 5. Evaluate names without changing cluster IDs

TF-IDF character n-grams (2–4) compare lowercased, whitespace-normalized names
inside the original clusters. `threshold` is exploratory, not a probability.
`max_clusters=None` evaluates every qualifying cluster.

The record table includes the best neighbor/score and missing-name status.
The cluster table includes supported-name fraction and component counts. A cluster
can have well-supported names but still contain multiple separate name families.
Missing names receive separate proposals, not automatic decisions.

In [6]:
NAME_THRESHOLD = 0.85
name_review = evaluate_clusters(
    fusion,
    name_column="name",
    record_id_column="record_id",
    threshold=NAME_THRESHOLD,
    top_n=10,
    complete_limit=100,
    max_clusters=None,
)
display(name_review.clusters)
display(name_review.records)
display(name_review.name_pairs)
display(name_review.proposed_splits)
show_file(name_review.export_html(OUTPUT_DIR / "synthetic_name_review.html", fusion=fusion))


cluster_id,record_count,usable_name_count,name_threshold,unsupported_name_count,supported_name_fraction,name_component_count,proposed_component_count,graph_complete,requires_review,review_reasons
u64,u64,u64,f64,u64,f64,u64,u64,bool,bool,list[str]
0,5,4,0.85,0,1.0,2,3,true,true,"[""multiple_name_components"", ""missing_or_unusable_names""]"


record_id,cluster_id,name,best_name_neighbor_id,best_name_similarity,name_status,has_neighbor_above_threshold,proposed_cluster_id
str,u64,str,str,f64,str,bool,str
"""A""",0,"""NORTH HARBOR CREDIT UNION""","""B""",1.0,"""supported""",true,"""name:0:0"""
"""B""",0,"""north harbor credit union""","""A""",1.0,"""supported""",true,"""name:0:0"""
"""C""",0,"""DESERT VALLEY FINANCE""","""D""",1.0,"""supported""",true,"""name:0:1"""
"""D""",0,"""DESERT VALLEY FINANCE""","""C""",1.0,"""supported""",true,"""name:0:1"""
"""E""",0,null,null,null,"""missing_name""",false,"""name:0:2"""


cluster_id,record_id_a,record_id_b,name_similarity,meets_threshold
u64,str,str,f64,bool
0,"""A""","""B""",1.0,true
0,"""A""","""C""",0.014333,false
0,"""A""","""D""",0.014333,false
0,"""B""","""C""",0.014333,false
0,"""B""","""D""",0.014333,false
0,"""C""","""D""",1.0,true


record_id,cluster_id,proposed_cluster_id,reason
str,u64,str,str
"""A""",0,"""name:0:0""","""name_similarity_proposal; requ…"
"""B""",0,"""name:0:0""","""name_similarity_proposal; requ…"
"""C""",0,"""name:0:1""","""name_similarity_proposal; requ…"
"""D""",0,"""name:0:1""","""name_similarity_proposal; requ…"
"""E""",0,"""name:0:2""","""name_similarity_proposal; requ…"


/Users/shumanti/Development/entity-fusion/test_data/_debug_outputs/transitive_notebook/synthetic_name_review.html

## 6. Explicitly accept a synthetic split

This decision is **manually specified**, not automatically accepted from name
proposals. We assume a reviewer confirmed A/B/E are one institution and C/D are
another; E's assignment uses external information because its name is missing.

Every member of an affected original cluster needs an assignment. Original cluster
IDs remain intact. `crossing_evidence` shows exactly which original match groups
now span multiple refined groups, including overrides of shared SSNs/licenses.
Persisted decisions follow stable source record IDs, not row positions. These toy
IDs are stable only for this synthetic fixture.

In [7]:
decisions = pl.DataFrame({
    "record_id": list("ABCDE"),
    "refined_cluster_id": ["FI-NORTH", "FI-NORTH", "FI-DESERT", "FI-DESERT", "FI-NORTH"],
    "reason": ["Synthetic review confirmed two institutions; E belongs to NORTH"] * 5,
})
decision_path = OUTPUT_DIR / "synthetic_accepted_splits.parquet"
decisions.write_parquet(decision_path)
split = apply_cluster_splits(
    fusion, pl.read_parquet(decision_path), record_id_column="record_id",
)
display(split.records)
display(split.crossing_evidence)
assert split.records.drop("refined_cluster_id").equals(fusion.records)
split.records.write_parquet(OUTPUT_DIR / "synthetic_refined_records.parquet")
split.crossing_evidence.write_parquet(OUTPUT_DIR / "synthetic_crossing_evidence.parquet")


record_id,name,ssn,driver_license,license_state,phone,country,email,cluster_id,refined_cluster_id
str,str,str,str,str,str,str,str,u64,str
"""A""","""NORTH HARBOR CREDIT UNION""","""SYNTH-1""","""L1""","""NY""","""555-0101""","""US""","""a@example.test""",0,"""FI-NORTH"""
"""B""","""north harbor credit union""","""SYNTH-1""","""L2""","""NY""","""555-0102""","""US""","""b@example.test""",0,"""FI-NORTH"""
"""C""","""DESERT VALLEY FINANCE""","""SYNTH-2""","""L2""","""NY""","""555-0102""","""US""","""c@example.test""",0,"""FI-DESERT"""
"""D""","""DESERT VALLEY FINANCE""","""SYNTH-3""","""L4""","""CA""","""555-0102""","""US""","""c@example.test""",0,"""FI-DESERT"""
"""E""",null,"""SYNTH-1""","""L1""","""NY""","""555-0101""","""US""","""a@example.test""",0,"""FI-NORTH"""
"""F""","""UNRELATED INSTITUTION""",null,null,null,null,null,null,5,"""original:5"""


group_id,cluster_id,row_positions,conditions,shared_values,refined_cluster_ids
u64,u64,list[u64],list[str],str,list[str]
4,0,"[1, 2]","[""driver_license [block_on: license_state]"", ""phone [block_on: country]""]","""{""country"": ""US"", ""driver_lice…","[""FI-DESERT"", ""FI-NORTH""]"


## 7. Configure the local real-data experiment

The file named `100k.parquet` actually contains 968,512 records. Start with the
sample below. Set `ROW_LIMIT=None` for the full file, and
`REVIEW_MAX_CLUSTERS=None` to evaluate every nonsingleton cluster. A previous full
run took about 44 seconds for name evaluation; runtime and memory depend on the
machine. HTML remains limited to the largest ten evaluated clusters.

The default exact rule is **name OR address, each blocked by city and state**.
Set `EXACT_MODE="strict"` for name AND address with those blockers.

The Parquet source lacks a stable record ID, so this notebook creates a row-based
ID for this extraction. Do not reuse it for durable manual decisions across
changed extracts. The real-data section makes no automatic membership changes.

In [8]:
INPUT_PATH = ROOT / "test_data" / "100k.parquet"
ROW_LIMIT = 10_000             # None = full file
REVIEW_MAX_CLUSTERS = 10       # None = all nonsingleton clusters
REAL_NAME_THRESHOLD = 0.85
EXACT_MODE = "blocked"        # "blocked" or "strict"
REPORT_TOP_N = 10


In [9]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Local dataset not found: {INPUT_PATH}")
if EXACT_MODE not in {"blocked", "strict"}:
    raise ValueError("EXACT_MODE must be blocked or strict")
if ROW_LIMIT is not None and (type(ROW_LIMIT) is not int or ROW_LIMIT < 1):
    raise ValueError("ROW_LIMIT must be a positive integer or None")

started = perf_counter()
scan = pl.scan_parquet(INPUT_PATH).select(
    "BorrowerName", "BorrowerAddress", "BorrowerCity", "BorrowerState",
)
source_rows = scan.select(pl.len()).collect().item()
if ROW_LIMIT is not None:
    scan = scan.head(ROW_LIMIT)
real_records = scan.collect().with_row_index("demo_record_id")
real_rule = {
    "and" if EXACT_MODE == "strict" else "or": [
        {"field": column, "block_on": ["BorrowerCity", "BorrowerState"]}
        for column in ("BorrowerName", "BorrowerAddress")
    ]
}
real_fusion = cluster_entities(real_records, match_rule=real_rule, return_details=True)
exact_seconds = perf_counter() - started
cluster_sizes = real_fusion.records.group_by("cluster_id").len(name="record_count").sort(
    ["record_count", "cluster_id"], descending=[True, False],
)
print(f"Source rows: {source_rows:,}; processed: {real_records.height:,}")
print(f"Load + exact clustering: {exact_seconds:.2f} seconds")
print(json.dumps(real_rule, indent=2))
display(cluster_sizes.head(10))
assert real_fusion.records.drop("cluster_id").equals(real_records)


Source rows: 968,512; processed: 10,000
Load + exact clustering: 0.09 seconds
{
  "or": [
    {
      "field": "BorrowerName",
      "block_on": [
        "BorrowerCity",
        "BorrowerState"
      ]
    },
    {
      "field": "BorrowerAddress",
      "block_on": [
        "BorrowerCity",
        "BorrowerState"
      ]
    }
  ]
}


cluster_id,record_count
u64,u32
258,9
6215,9
6829,8
4923,7
5,6
314,6
347,6
4280,6
5755,6


## 8. Review real clusters and inspect flags

TF-IDF is fit over the selected evaluation corpus, so changing the sample or
cluster limit can change scores. `graph_complete=False` means top-N retrieval may
omit qualifying links. Name components use transitive connections, not an
all-pairs minimum. Flags indicate cases to inspect, not confirmed incorrect links.

In [10]:
started = perf_counter()
real_review = evaluate_clusters(
    real_fusion,
    name_column="BorrowerName",
    record_id_column="demo_record_id",
    threshold=REAL_NAME_THRESHOLD,
    top_n=10,
    complete_limit=100,
    max_clusters=REVIEW_MAX_CLUSTERS,
)
name_seconds = perf_counter() - started
print(f"Name evaluation: {name_seconds:.2f} seconds")
print(f"Evaluated clusters: {real_review.clusters.height:,}; records: {real_review.records.height:,}")
print(f"Flagged clusters: {real_review.clusters['requires_review'].sum():,}")
display(real_review.clusters.head(20))
display(real_review.records.group_by("name_status").len())


Name evaluation: 0.01 seconds
Evaluated clusters: 10; records: 69
Flagged clusters: 10


cluster_id,record_count,usable_name_count,name_threshold,unsupported_name_count,supported_name_fraction,name_component_count,proposed_component_count,graph_complete,requires_review,review_reasons
u64,u64,u64,f64,u64,f64,u64,u64,bool,bool,list[str]
258,9,9,0.85,1,0.888889,5,5,true,true,"[""multiple_name_components"", ""names_without_threshold_support""]"
6215,9,9,0.85,9,0.0,9,9,true,true,"[""multiple_name_components"", ""names_without_threshold_support""]"
6829,8,8,0.85,2,0.75,5,5,true,true,"[""multiple_name_components"", ""names_without_threshold_support""]"
4923,7,7,0.85,7,0.0,7,7,true,true,"[""multiple_name_components"", ""names_without_threshold_support""]"
5,6,6,0.85,6,0.0,6,6,true,true,"[""multiple_name_components"", ""names_without_threshold_support""]"
314,6,6,0.85,0,1.0,3,3,true,true,"[""multiple_name_components""]"
347,6,6,0.85,6,0.0,6,6,true,true,"[""multiple_name_components"", ""names_without_threshold_support""]"
4280,6,6,0.85,2,0.666667,4,4,true,true,"[""multiple_name_components"", ""names_without_threshold_support""]"
5755,6,6,0.85,6,0.0,6,6,true,true,"[""multiple_name_components"", ""names_without_threshold_support""]"


name_status,len
str,u32
"""supported""",24
"""below_threshold""",45


In [11]:
real_review.records.filter(pl.col("cluster_id") == 258)

record_id,cluster_id,name,best_name_neighbor_id,best_name_similarity,name_status,has_neighbor_above_threshold,proposed_cluster_id
u32,u64,str,u32,f64,str,bool,str
258,258,"""BETHEL CONTRACTING LLC""",259,1.0,"""supported""",true,"""name:258:0"""
259,258,"""BETHEL CONTRACTING LLC""",258,1.0,"""supported""",true,"""name:258:0"""
980,258,"""BETHEL ENVIRONMENTAL SOLUTIONS…",981,1.0,"""supported""",true,"""name:258:1"""
981,258,"""BETHEL ENVIRONMENTAL SOLUTIONS…",980,1.0,"""supported""",true,"""name:258:1"""
1708,258,"""BETHEL NATIVE CORPORATION""",2172,0.281009,"""below_threshold""",false,"""name:258:2"""
2172,258,"""BETHEL SOLUTIONS INC""",2173,1.0,"""supported""",true,"""name:258:3"""
2173,258,"""BETHEL SOLUTIONS INC""",2172,1.0,"""supported""",true,"""name:258:3"""
2241,258,"""BETHEL FEDERAL SERVICES LLC""",2242,1.0,"""supported""",true,"""name:258:4"""
2242,258,"""BETHEL FEDERAL SERVICES LLC""",2241,1.0,"""supported""",true,"""name:258:4"""


In [12]:
# Change this ID to inspect another evaluated cluster.
INSPECT_CLUSTER_ID = real_review.clusters["cluster_id"][0] if real_review.clusters.height else None
if INSPECT_CLUSTER_ID is not None:
    display(real_review.records.filter(pl.col("cluster_id") == INSPECT_CLUSTER_ID))
    display(real_fusion.match_groups.filter(pl.col("cluster_id") == INSPECT_CLUSTER_ID))
    display(real_review.name_pairs.filter(pl.col("cluster_id") == INSPECT_CLUSTER_ID).sort(
        "name_similarity", descending=True,
    ).head(30))


record_id,cluster_id,name,best_name_neighbor_id,best_name_similarity,name_status,has_neighbor_above_threshold,proposed_cluster_id
u32,u64,str,u32,f64,str,bool,str
258,258,"""BETHEL CONTRACTING LLC""",259,1.0,"""supported""",true,"""name:258:0"""
259,258,"""BETHEL CONTRACTING LLC""",258,1.0,"""supported""",true,"""name:258:0"""
980,258,"""BETHEL ENVIRONMENTAL SOLUTIONS…",981,1.0,"""supported""",true,"""name:258:1"""
981,258,"""BETHEL ENVIRONMENTAL SOLUTIONS…",980,1.0,"""supported""",true,"""name:258:1"""
1708,258,"""BETHEL NATIVE CORPORATION""",2172,0.281009,"""below_threshold""",false,"""name:258:2"""
2172,258,"""BETHEL SOLUTIONS INC""",2173,1.0,"""supported""",true,"""name:258:3"""
2173,258,"""BETHEL SOLUTIONS INC""",2172,1.0,"""supported""",true,"""name:258:3"""
2241,258,"""BETHEL FEDERAL SERVICES LLC""",2242,1.0,"""supported""",true,"""name:258:4"""
2242,258,"""BETHEL FEDERAL SERVICES LLC""",2241,1.0,"""supported""",true,"""name:258:4"""


group_id,cluster_id,row_positions,conditions,shared_values
u64,u64,list[u64],list[str],str
53,258,"[258, 259]","[""BorrowerName [block_on: BorrowerCity, BorrowerState]""]","""{""BorrowerCity"": ""anchorage"", …"
54,258,"[258, 981, … 2241]","[""BorrowerAddress [block_on: BorrowerCity, BorrowerState]""]","""{""BorrowerAddress"": ""2605 Dena…"
55,258,"[259, 2173, 2242]","[""BorrowerAddress [block_on: BorrowerCity, BorrowerState]""]","""{""BorrowerAddress"": ""2605 DENA…"
199,258,"[980, 981]","[""BorrowerName [block_on: BorrowerCity, BorrowerState]""]","""{""BorrowerCity"": ""anchorage"", …"
381,258,"[2172, 2173]","[""BorrowerName [block_on: BorrowerCity, BorrowerState]""]","""{""BorrowerCity"": ""anchorage"", …"
390,258,"[2241, 2242]","[""BorrowerName [block_on: BorrowerCity, BorrowerState]""]","""{""BorrowerCity"": ""anchorage"", …"


cluster_id,record_id_a,record_id_b,name_similarity,meets_threshold
u64,u32,u32,f64,bool
258,258,259,1.0,true
258,980,981,1.0,true
258,2172,2173,1.0,true
258,2241,2242,1.0,true
258,980,2172,0.630551,false
…,…,…,…,…
258,981,2242,0.257794,false
258,258,1708,0.250567,false
258,259,1708,0.250567,false


## 9. Export full tables and bounded HTML

Parquet files retain all evaluated rows; the name HTML contains only the largest
`REPORT_TOP_N` evaluated clusters. The name dashboard includes exact shared-value evidence and connection paths; a separate exact report adds graphs.
Original labeled records, exact evidence, configuration, and proposed splits are
saved together. No proposed real-data split is accepted in this notebook.

In [13]:
from dataclasses import replace

run_dir = OUTPUT_DIR / f"{EXACT_MODE}_{real_records.height}_review_{REVIEW_MAX_CLUSTERS or 'all'}"
run_dir.mkdir(parents=True, exist_ok=True)
real_fusion.records.write_parquet(run_dir / "labeled_records.parquet")
real_fusion.match_groups.write_parquet(run_dir / "exact_match_groups.parquet")
for label in ("records", "clusters", "name_pairs", "proposed_splits"):
    getattr(real_review, label).write_parquet(run_dir / f"{label}.parquet")

report_ids = real_review.clusters["cluster_id"].head(REPORT_TOP_N).to_list()
report_review = replace(
    real_review,
    **{label: getattr(real_review, label).filter(pl.col("cluster_id").is_in(report_ids))
       for label in ("records", "clusters", "name_pairs", "proposed_splits")},
    configuration={**real_review.configuration, "html_scope": f"largest {REPORT_TOP_N} evaluated clusters"},
)
show_file(report_review.export_html(run_dir / "name_review.html", fusion=real_fusion))
show_file(real_fusion.export_html(
    run_dir / "exact_review.html", top_n=REPORT_TOP_N,
    record_id_column="demo_record_id", conflict_columns=["BorrowerName"],
))
summary = {
    "input_path": str(INPUT_PATH), "source_rows": source_rows,
    "processed_rows": real_records.height, "exact_rule": real_rule,
    "load_and_exact_seconds": exact_seconds, "name_evaluation_seconds": name_seconds,
    "review_configuration": real_review.configuration,
    "flagged_clusters": real_review.clusters["requires_review"].sum(),
    "manual_splits_applied": False,
    "id_note": "Generated row IDs are extraction-specific, not durable source IDs",
}
(run_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved outputs:", run_dir)


/Users/shumanti/Development/entity-fusion/test_data/_debug_outputs/transitive_notebook/blocked_10000_review_10/name_review.html

/Users/shumanti/Development/entity-fusion/test_data/_debug_outputs/transitive_notebook/blocked_10000_review_10/exact_review.html

Saved outputs: /Users/shumanti/Development/entity-fusion/test_data/_debug_outputs/transitive_notebook/blocked_10000_review_10


## Further experiments

- Raise/lower `REAL_NAME_THRESHOLD` and compare flags and name components.
- Compare blocked OR with strict AND exact rules.
- Use `ROW_LIMIT=None`, `REVIEW_MAX_CLUSTERS=None` for full-file evaluation.
- Inspect exact evidence before accepting any proposed name split, especially when
  authoritative identifiers cross the proposed boundary.
- Use stable source IDs and an explicit decision table for reviewed real-data splits.

See [Transitive_Fusion_README.md](./Transitive_Fusion_README.md) for API details,
review-table definitions, and earlier benchmark measurements.